In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [ ]:
import os

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import (
    HfApi,
    create_repo,
    get_token,
    notebook_login,
)
from transformers import AutoModelForCausalLM, AutoProcessor


In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"Total GPUs Detected: {gpu_count}\n")

    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"--- GPU {idx} ---")
        print(f"  Name: {props.name}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(
            f"  Allocated VRAM: {torch.cuda.memory_allocated(idx) / (1024**3):.2f} GB"
        )
        print(
            f"  Free VRAM: {(props.total_memory - torch.cuda.memory_reserved(idx)) / (1024**3):.2f} GB\n"
        )
else:
    print("No CUDA-compatible GPUs found.")

In [ ]:
notebook_login()

In [ ]:
MODEL_ID = "dots-studio/dots.mocr"
HF_USER = "your-huggingface-username" # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [ ]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


In [ ]:
# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1200,                            # Optimization iterations per block; balances calibration time and accuracy
    gradient_accumulate_steps=4,           # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                           # Target pretrained model to be quantized
    tokenizer=tokenizer,                   # Tokenizer used to process text calibration samples
    processor=processor,                   # Processor for multimodal models (vision/audio), if present
    scheme="NVFP4",                        # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,                # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                          # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=2,                          # Batch size for calibration; kept < 8 to avoid slow execution/high memory overhead
    seqlen=4096,                           # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,               # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,             # Compiles forward passes using torch.compile to accelerate calibration
    quant_nontext_module=False,            # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision
    ignore_layers="vision_tower,lm_head",  # Excludes and visual modules from quantizationx.difference(y)
)


In [ ]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Serializes weights to 3 formats simultaneously to guarantee compatibility:
    # - 'auto_round': Reference AutoRound format (vLLM GPTQ/Marlin, Transformers / native loader)
    # - 'auto_gptq': AutoGPTQ tensor layout (vLLM GPTQ/Marlin, TGI)
    # Use 'llm_compressor' for the compressed-tensors schema, recommended for NVFP4 inference with vLLM / Neural Magic
    format="llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

In [ ]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns=[
                "*.safetensors",
                "*.bin",
                "*.pt",
                "*.gguf",
                "*.json",
                "*.model",
                "*.onnx",
            ],
            commit_message="Update model weights and config",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-nvfp-w4g16"),
        f"{base_name}-NVFP4",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")